# Profilering af regionalt netbelastning og udfald med PROC CHART

## Ledelsesresume

En netdriftsanalytiker bruger PROC CHART til at profilere en syntetisk stikprøve af målaflæsninger fra fordelingskredse på tværs af fire forsyningsregioner og fire produktionskilder. Notebooken gennemgår lodret søjlediagram, vandret søjlediagram, blokdiagram og cirkeldiagram for at opsummere produktionsmikset, sammenligne gennemsnitlig og samlet belastning pr. region, afdække aftenens efterspørgselstop pr. time og rangordne udfaldsminutter pr. kilde — den slags hurtige tekstbaserede udforskning, som et energi- og forsyningsteam kører, før det forpligter sig til en grafisk rapport. DATA-trinnet anmoder om 1.200 rækker; denne notebook blev gengivet i ulicenseret tilstand, som begrænser arbejdsdatasættet til de første 100 aflæsninger, så hvert diagram nedenfor opsummerer den 100-rækkers stikprøve.

## Datakilder

| Datasæt | Rækker | Beskrivelse |
|---------|------|-------------|
| `grid_ops` | 100 (syntetisk stikprøve) | Én række pr. målaflæsning fra en fordelingskreds på et regionalt net, genereret inline med `call streaminit(20260531)` og `rand()`. DATA-trinnets løkke beder om 1.200 aflæsninger, men ulicenseret tilstand begrænser det gemte datasæt til 100 observationer, så diagrammerne nedenfor beskriver de 100 rækker. |

# Profilering af regionalt netbelastning og udfald med PROC CHART

PROC CHART producerer tegnbaserede søjle-, blok- og cirkeldiagrammer direkte i listingen — ingen ODS Graphics-enhed kræves. Det gør den til et hurtigt førstehåndsudforskningsværktøj for et netdriftsteam, der vil *se* formen på sine belastnings- og pålidelighedsdata, før det bygger polerede GCHART- eller SGPLOT-visualiseringer.

I denne notebook:

1. Genererer vi en syntetisk måned af målaflæsninger fra fordelingskredse for en regional netoperatør.
2. Diagrammerer **produktionsmikset** (andel af aflæsninger pr. kilde).
3. Sammenligner **gennemsnitlig og samlet leveret belastning** på tværs af forsyningsregioner.
4. Afdækker **aftenens efterspørgselstop** pr. time på dagen.
5. Bruger et **blokdiagram** til at krydse region mod produktionskilde.
6. Rangordner **udfaldsminutter** pr. kilde for at finde den mindst pålidelige forsyning.

Hver sætning og indstilling nedenfor er standard SAS 9.4 PROC CHART-syntaks.

## Trin 1 — Generér de syntetiske driftsdata

DATA-trinnet nedenfor fabrikerer målaflæsninger i en løkke på 1.200 iterationer. Hver aflæsning tildeles en forsyningsregion, en produktionskilde (Gas dominerer, med Wind, Solar og Hydro som resten) og en time på dagen. Belastningen er højere i aftenvinduet 17:00–21:00 (og vi markerer disse aflæsninger som spidsbelastning), og vi trækker udfaldsminutter fra en Poisson-fordeling. `call streaminit` fastlægger det tilfældige startpunkt, så data er reproducerbare.

NOTE i loggen viser det praktiske resultat: denne kørsel er i ulicenseret tilstand, så det gemte `grid_ops`-datasæt er begrænset til de første 100 af disse aflæsninger (100 rækker, 7 kolonner). Hvert diagram, der følger, opsummerer den 100-rækkers stikprøve, og hver PROC CHART-log bekræfter, at den læste 100 rækker.

In [1]:
/* Synthetic feeder-circuit operations for a regional grid operator */
data grid_ops;
    CALL streaminit(20260531);
    LÆNGDE region $12 source $9 peak_flag $3;
    TABEL regions[4] $12 _temporary_
        ('North','South','East','West');
    GØR meter_id = 1 TIL 1200;
        r = ceil(4 * rand('uniform'));
        region = regions[r];
        u = rand('uniform');
        HVIS u < 0.42 SÅ source = 'Gas';
        ELLERS HVIS u < 0.70 SÅ source = 'Wind';
        ELLERS HVIS u < 0.88 SÅ source = 'Solar';
        ELLERS source = 'Hydro';
        hour = floor(24 * rand('uniform'));
        BASE = 18 + 5 * (region = 'North') + 3 * (region = 'West');
        HVIS hour >= 17 AND hour <= 21 SÅ GØR;
            load_mwh = BASE + 12 + 6 * rand('normal');
            peak_flag = 'Yes';
        SLUT;
        ELLERS GØR;
            load_mwh = BASE + 4 * rand('normal');
            peak_flag = 'No';
        SLUT;
        HVIS load_mwh < 0 SÅ load_mwh = 0;
        outage_min = rand('poisson', 2.5);
        UDDATA;
    SLUT;
    FJERN r u BASE;
KØR;

NOTE: DATA grid_ops

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote grid_ops (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds


## Trin 2 — Produktionsmiks

Hvilken andel af aflæsningerne står hver produktionskilde for? Et lodret søjlediagram med `TYPE=PERCENT` besvarer dette direkte: søjlehøjderne er procentdelen af alle observationer, der falder i hver kildekategori. Fordi `source` er en tegnvariabel, behandler PROC CHART automatisk dens værdier som diskrete kategorier.

In [2]:
PROCEDURE chart data=grid_ops;
    VBAR source / type=PERCENT;
KØR;


Percent of SOURCE

         |              *****       
         |              *****       
   40.00 +              *****       
         |              *****       
         |              *****       
   30.00 +              *****       
         |       *****  *****       
         |       *****  *****       
   20.00 +       *****  *****       
         |       *****  *****       
         |*****  *****  *****       
         |*****  *****  *****  *****
   10.00 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          Hydro  Wind    Gas   Solar
                    SOURCE


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Trin 3 — Gennemsnitlig leveret belastning pr. region

Nu skifter vi fra at tælle til at opsummere en måling. Ved at navngive `load_mwh` i `SUMVAR=` med `TYPE=MEAN` bliver søjlelængden den gennemsnitlige belastning pr. region, og vi anmoder om statistikkolonnerne eksplicit: `MEAN` udskriver gennemsnittet ved siden af hver søjle, og `CFREQ` tilføjer en kumuleret frekvenskolonne. North bærer den højeste gennemsnitlige leverede belastning (gennemsnit omkring 28,0 MWh), i overensstemmelse med det regionale forskydning indbygget i data, mens South er lavest (omkring 19,8 MWh).

Vi videregiver også `ASCENDING` med den hensigt at ordne søjlerne fra laveste til højeste gennemsnit. Bemærk i outputtet, at søjlerne **ikke** omordnes — de vises i kategorirækkefølge (West, South, East, North), med gennemsnit 24,2, 19,8, 21,7, 28,0, der ikke løber fra korteste til længste. Omordning af søjler efter diagramstatistikken er endnu ikke koblet op i denne PROC CHART-implementering, så `ASCENDING`/`DESCENDING` accepteres, men har i øjeblikket ingen effekt; læs rangordningen fra den udskrevne `Mean`-kolonne i stedet.

In [3]:
PROCEDURE chart data=grid_ops;
    HBAR region / SUMVAR=load_mwh type=mean
                  cfreq mean ascending;
KØR;


Mean of REGION

REGION                                                  Mean           N     Percent
                                                                                    
  West  **********************************             24.17       24.17       23.00
 South  ****************************                   19.80       43.97       21.00
  East  *******************************                21.72       65.69       29.00
 North  ****************************************       28.03       93.72       27.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Trin 4 — Samlet belastning pr. region

Her gør `TYPE=SUM` hver søjle til den *samlede* leverede belastning for regionen frem for gennemsnittet, så de højere søjler markerer de regioner, der leverer mest samlet energi på tværs af de stikprøvetagne aflæsninger. North fører igen på samlet leveret belastning.

Sætningen anmoder også om `SUBGROUP=peak_flag`, som beder PROC CHART om at opdele hver søjle efter, om aflæsningen faldt i aftenens spidsbelastningsvindue. I SAS segmenterer dette hver søjle med en distinkt glyf pr. undergruppeniveau og udskriver en forklaring. Denne implementering gengiver endnu ikke undergruppesegmentering (et dokumenteret kapabilitetshul), så søjlerne kommer ud som massive `*`-forløb uden opdeling pr. segment — søjle*totalerne* er korrekte, men opdelingen mellem spids og ikke-spids vises ikke her. For at se, hvor meget belastning der lander i spidstimerne, kan du bruge time-på-dagen-visningen i Trin 5.

In [4]:
PROCEDURE chart data=grid_ops;
    VBAR region / SUMVAR=load_mwh type=sum
                  SUBGROUP=peak_flag;
KØR;


Sum of REGION

         |                     *****
         |                     *****
         |                     *****
     600 +              *****  *****
         |*****         *****  *****
         |*****         *****  *****
         |*****         *****  *****
     400 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
     200 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          West   South  East   North
                    REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Trin 5 — Belastningsformen henover dagen

`hour` er kontinuert, så vi fastlægger selv opdelingen med en eksplicit `MIDPOINTS=`-liste ved 4-timers centre (2, 6, 10, 14, 18, 22). Hver søjle viser den gennemsnitlige leverede belastning for aflæsninger nær den time. Søjlen centreret om 18 bør skille sig ud — det er aftenens efterspørgselstop, vi byggede ind i data.

In [5]:
PROCEDURE chart data=grid_ops;
    VBAR hour / SUMVAR=load_mwh type=mean
                midpoints=2 6 10 14 18 22;
KØR;


Mean of HOUR

         |                            *****       
         |                            *****       
   30.00 +                            *****       
         |                            *****  *****
         |                            *****  *****
         |              *****         *****  *****
   20.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
   10.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |                                        
         +----------------------------------------+
            2      6     10     14     18     22  
                            HOUR


NOTE: PROC CHART data=grid_ops


## Trin 6 — Region efter produktionskilde (blokdiagram)

Et BLOCK-diagram gengiver en lille tovejstabel som et gitter af blokke. Med `GROUP=source` og `SUMVAR=load_mwh / TYPE=MEAN` krydser diagrammet hver region mod den produktionskilde, der betjener den, med blokhøjde proportional med gennemsnitsbelastningen — en kompakt måde at få øje på, hvilke region/kilde-kombinationer der bærer den tungeste gennemsnitlige belastning.

In [6]:
PROCEDURE chart data=grid_ops;
    BLOCK region / SUMVAR=load_mwh type=mean
                   GROUP=source;
KØR;


Block chart of Mean of REGION by SOURCE

                          /####\
  /####\                  /####\
  /####\          /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  +----+  +----+  +----+  +----+
   West   South    East   North 
             REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Trin 7 — Produktionsmiks som cirkeldiagram

Den samme kildeandelsinformation fra Trin 2, tegnet som en cirkel. PIE med `TYPE=PERCENT` dimensionerer hvert udsnit efter dets procentdel af de samlede aflæsninger og udskriver en forklaring med udsnitsprocenter ved siden af figuren.

In [7]:
PROCEDURE chart data=grid_ops;
    PIE source / type=PERCENT;
KØR;


Pie chart of SOURCE

          SOURCE     Percent   Percent  Slice
----------------  ----------  --------  ------------------------------
           Hydro       14.00     14.0%  ####
            Wind       28.00     28.0%  ********
             Gas       45.00     45.0%  ++++++++++++++
           Solar       13.00     13.0%  OOOO


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Trin 8 — Udfaldsminutter pr. kilde

Til sidst rangordner vi pålideligheden. `SUMVAR=outage_min` med `TYPE=SUM` totaliserer udfaldsminutter pr. kilde. Vi videregiver `DESCENDING` for at forsøge at flyde den dårligst præsterende kilde til toppen, men som i Trin 3 omordnes søjlerne ikke — de udskrives i kategorirækkefølge (Hydro, Wind, Gas, Solar), og søjleomordning er endnu ikke implementeret. Læs rangordningen fra den udskrevne `Sum`-kolonne: Gas står for de fleste samlede udfaldsminutter i denne stikprøve (122), et godt stykke foran Wind (64), Solar (43) og Hydro (38). Det følger produktionsmikset — Gas betjener de fleste aflæsninger, så den akkumulerer de fleste udfaldsminutter samlet set.

In [8]:
PROCEDURE chart data=grid_ops;
    HBAR source / SUMVAR=outage_min type=sum FALDENDE;
KØR;


Sum of SOURCE

SOURCE                                                   Sum        Cum.     Percent
                                                                     Sum            
 Hydro  ************                                      38          38       14.00
  Wind  *********************                             64         102       28.00
   Gas  ****************************************         122         224       45.00
 Solar  **************                                    43         267       13.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Fortolkning af resultaterne

At læse diagrammerne sammen giver driftsteamet et hurtigt situationsbillede (over den 100-rækkers stikprøve, som denne kørsel indfangede):

- **Produktionsmiks (Trin 2 og 7).** Gas bærer den største andel af aflæsninger (omkring 45 %), med Wind som nummer to (omkring 28 %) og Hydro og Solar bagest (omkring 14 % og 13 %) — det lodrette søjlediagram og cirklen fortæller den samme historie på to måder, en nyttig fornuftskontrol.
- **Belastning pr. region (Trin 3 og 4).** Gennemsnitsbelastningens HBAR viser North med den højeste gennemsnitlige leverede belastning (gennemsnit ~28 MWh) og South den laveste (~20 MWh), i overensstemmelse med den regionale forskydning indbygget i data. SUM-diagrammet bekræfter, at North også leverer mest samlet energi.
- **Daglig belastningsform (Trin 5).** Midtpunktssøjlen centreret om time 18 er tydeligt den højeste, hvilket bekræfter efterspørgselstoppet 17:00–21:00, vi byggede ind i data — præcis dér, hvor et forsyningsselskab ville fokusere efterspørgselsstyring og kapacitetsplanlægning.
- **Pålidelighed (Trin 8).** Totalisering af udfaldsminutter pr. kilde afdækker Gas som den største bidragyder til nedetid i denne stikprøve (122 minutter), det naturlige næste mål for vedligeholdelsesgennemgang — selvom det for det meste afspejler, at Gas betjener de fleste aflæsninger.

To visningsindstillinger brugt her — `ASCENDING`/`DESCENDING`-søjleomordning (Trin 3 og 8) og `SUBGROUP=`-søjlesegmentering (Trin 4) — accepteres af parseren, men gengives endnu ikke af denne PROC CHART-implementering, så rangordninger og opdelinger læses fra de udskrevne statistikkolonner frem for fra søjlerækkefølgen eller skyggelægningen.

PROC CHART er kun tegnbaseret output, så for bestyrelsesklare visualiseringer ville teamet flytte de samme visninger til PROC GCHART eller PROC SGPLOT. Men som et første gennemløb uden opsætning over et frisk udtræk besvarer disse diagrammer de operationelle spørgsmål — miks, størrelsesorden og timing — på sekunder.